# HX-Cascade v3 — step 2: sweep, calibrate, evaluate (GPU notebook)

Settings: **Accelerator: GPU (T4 or P100)**, **Internet: On**. Add the dataset notebook's output with
**Add Input > Your Work** and set `DATASET_INPUT` below. Run with **Save & Run All (Commit)**.

1. Every `SWEEP` config is trained on the training maps; the winner is the one with the highest mean average
   precision (900/1800/3600 s) **on the validation maps only**.
2. The winner is calibrated (one temperature per horizon) and its alert thresholds chosen, on validation maps.
3. It is evaluated **once** on the test maps and, separately, on the live map, against the deterministic cascade,
   the forecast-threshold rule and v2, with cluster-bootstrap 95% intervals. Nothing is tuned on these.

Output: `/kaggle/working/hx_cascade_v3.zip` = model.pt, feature_norm.json, calibration.json, ood_stats.json,
eval.json, training.json, manifest.json (commit, dataset version, seeds, file hashes, library versions).
Everything here is simulation-only evidence (03 §8.4).

In [ ]:
# ============================ PARAMETERS ============================
# --- where the code comes from ----------------------------------------------
# "git": clone REPO_URL at REPO_REF (Settings > Internet must be ON)
# "dataset": the repo uploaded as a zip Kaggle Dataset, mounted at REPO_DATASET
CODE_SOURCE  = "git"
REPO_URL     = "https://github.com/GAURAV-MORE28/EventFlow-AI.git"
REPO_REF     = "gaurav"
REPO_COMMIT  = None        # pin an exact commit SHA here for a reproducible rerun
REPO_DATASET = "/kaggle/input/eventflow-ai/EventFlow-AI"

# --- data -----------------------------------------------------------------------
DATASET_INPUT   = "/kaggle/input/<your-dataset-notebook>/cascade_v3"   # <- set this
DATASET_VERSION = "cascade-v3-dataset notebook, version 1"            # recorded in the manifest

# --- sweep (each entry overrides BASE; the winner is chosen on validation maps) ---
BASE  = {"hidden": 64, "layers": 3, "heads": 4, "dropout": 0.1, "lr": 2e-3, "batch_size": 128, "pos_weight": "sqrt"}
SWEEP = [
    {},                                        # the base config
    {"hidden": 128},
    {"layers": 2},
    {"pos_weight": "none"},
    {"pos_weight": "linear"},
    {"lr": 5e-4},
]
EPOCHS, PATIENCE = 30, 5
TTC_WEIGHT       = 0.5
V2_THRESHOLD     = 0.6      # v2's deployed gnn_min_probability (Backend/config.yaml)
BOOTSTRAP        = 200
TRAIN_SEED       = 42
DEVICE           = "auto"   # auto | cpu | cuda
MODEL_VERSION    = "hx_cascade_v3"
# =====================================================================

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/EventFlow-AI")          # outside /kaggle/working, so the code is not saved as output

# 1. The code, at a recorded commit.
if not REPO.exists():
    if CODE_SOURCE == "dataset":
        shutil.copytree(REPO_DATASET, REPO)
    else:
        url = REPO_URL
        try:   # private repo only: Add-ons > Secrets > GITHUB_TOKEN (never paste a token into a cell)
            from kaggle_secrets import UserSecretsClient
            url = url.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--quiet", "--branch", REPO_REF, url, str(REPO)], check=True)
        if REPO_COMMIT:
            subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True,
                            check=True).stdout.strip()
except Exception:
    COMMIT = REPO_COMMIT or None     # zip upload: no .git, so set REPO_COMMIT by hand
print("code commit:", COMMIT)

# 2. Dependencies. Kaggle ships torch and numpy: keep them, add the backend's
#    pinned packages (including torch_geometric) on top.
reqs = [l.split("#")[0].strip() for l in (REPO / "Backend" / "requirements.txt").read_text().splitlines()]
reqs = [r for r in reqs if r and r.split("==")[0].split("[")[0].lower() not in {"torch", "numpy", "pytest"}]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)

# 3. Environment check: PyG must work with THIS torch, and the forecaster must
#    run in the same mode as the live backend (no Chronos TSFM installed).
import torch, torch_geometric, numpy as np
from torch_geometric.nn import RGCNConv, TransformerConv
x = torch.randn(5, 8); ei = torch.tensor([[0, 1, 2], [1, 2, 3]])
TransformerConv(8, 4, heads=2, edge_dim=3)(x, ei, torch.randn(3, 3))
RGCNConv(8, 8, num_relations=2)(x, ei, torch.tensor([0, 1, 0]))
if importlib.util.find_spec("chronos") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "chronos-forecasting"], check=True)
    raise SystemExit("chronos was installed and has been removed: restart the kernel and run all again")
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | torch_geometric {torch_geometric.__version__} "
      f"| numpy {np.__version__} | cuda {torch.cuda.is_available()} | cores {os.cpu_count()}")
print("environment OK")

In [ ]:
DATA = Path(DATASET_INPUT)
summary = json.loads((DATA / "dataset.json").read_text())
print(json.dumps({k: summary.get(k) for k in ("source_commit", "complete", "maps", "runs")}, indent=1))
if summary.get("source_commit") and COMMIT and summary["source_commit"] != COMMIT:
    print(f"NOTE: dataset made at {summary['source_commit'][:8]}, training code at {COMMIT[:8]}")

BUNDLE = WORK / MODEL_VERSION
flags = {"--hidden": BASE["hidden"], "--layers": BASE["layers"], "--heads": BASE["heads"],
         "--dropout": BASE["dropout"], "--lr": BASE["lr"], "--batch-size": BASE["batch_size"],
         "--pos-weight": BASE["pos_weight"], "--epochs": EPOCHS, "--patience": PATIENCE,
         "--ttc-weight": TTC_WEIGHT, "--v2-threshold": V2_THRESHOLD, "--bootstrap": BOOTSTRAP,
         "--seed": TRAIN_SEED, "--device": DEVICE, "--version": MODEL_VERSION}
cmd = [sys.executable, "-m", "ML.training.train_v3", "--data", str(DATA), "--out", str(BUNDLE), "--force",
       "--sweep", json.dumps(SWEEP), "--dataset-version", DATASET_VERSION]
cmd += [str(v) for kv in flags.items() for v in kv]
if COMMIT:
    cmd += ["--source-commit", COMMIT]
t0 = time.time()
subprocess.run(cmd, cwd=REPO, check=True)
print(f"training step {(time.time() - t0) / 60:.0f} min")

In [ ]:
training = json.loads((BUNDLE / "training.json").read_text())
print("sweep (validation maps only):")
for row in sorted(training["sweep"], key=lambda r: -r["val_mean_ap"]):
    print(f"  val mean AP {row['val_mean_ap']:.4f}  epochs {row['epochs']:>2}  {row['seconds'] / 60:5.1f} min  {row['config']}")
print("selected:", training["config"])
print("timing:", training["timing"])

report = json.loads((BUNDLE / "eval.json").read_text())
cal = json.loads((BUNDLE / "calibration.json").read_text())
print("\ntemperatures", cal["temperature"], "| on search bound (= not fitted)", cal["at_search_bound"])
print("alert thresholds", cal["alert_threshold"])
print("caveats", report.get("caveats"))

def cell(res, key):
    v = res["point"]
    for k in key.split("."):
        v = v.get(k) if isinstance(v, dict) else None
    ci = res["ci95"].get(key)
    return "-" if v is None else f"{v:.3f}" + (f" [{ci[0]:.3f},{ci[1]:.3f}]" if ci else "")

cols = ["3600.precision", "3600.recall", "3600.average_precision", "3600.ece",
        "ttc_mae_sec", "lead_time_sec", "event_recall"]
for split in ("val", "test", "live"):
    if split not in report:
        continue
    r = report[split]
    print(f"\n== {split}: {r['maps']} maps, {r['runs']} runs ==")
    print(f"{'predictor':24s}" + "".join(f"{c:>30s}" for c in cols))
    for name, res in r["results"].items():
        print(f"{name:24s}" + "".join(f"{cell(res, c):>30s}" for c in cols))
print("\nswap criterion (test maps):", json.dumps(report.get("swap_criterion_test"), indent=1))

In [ ]:
# Verify every file against the manifest's SHA-256, then zip for download.
sys.path.insert(0, str(REPO))
from ML.manifest import load_manifest
m = load_manifest(BUNDLE)
print(m["model_version"], "| commit", m["source_commit"], "| dataset", m["dataset"]["version"])
print({k: v["sha256"][:12] for k, v in m["files"].items()})
archive = shutil.make_archive(str(WORK / MODEL_VERSION), "zip", BUNDLE)
print("download from the Output tab:", archive)